In [1]:
import subprocess, sys, json, glob, os
import pandas as pd

In [2]:
OUT = "/home/admins/rebuild_workspace/models_buildup"
CWD = "/home/admins/lip_codebase_clean/src/training"
os.makedirs(OUT, exist_ok=True)

In [ ]:
CONFIGS  = ["a0_1conv16","a1_1conv32","a2_2conv","a3_2conv_d128",
            "a4_baseline","a5_3conv","a6_3conv_narrow"]
DATASETS = ["d90","d138","d186","d234","d330"]
SEEDS    = [11, 22, 33]

for name in CONFIGS:
    for dkey in DATASETS:
        for seed in SEEDS:
            r = subprocess.run(
                [sys.executable, "cnn_systematic_buildup_worker.py", name, dkey, str(seed)],
                capture_output=True, text=True, cwd=CWD)
            print(r.stdout.strip() or r.stderr.strip()[-600:])

rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/result_*.json"))]
res_b = pd.DataFrame(rows)
res_b.to_csv(f"{OUT}/buildup_log.csv", index=False)

print("\n--- best_val by config x dataset ---")
print(res_b.pivot_table(index='config', columns='dataset',
                        values='best_val_acc', aggfunc='mean').round(4).to_string())
print("\n--- final_val by config x dataset ---")
print(res_b.pivot_table(index='config', columns='dataset',
                        values='final_val_acc', aggfunc='mean').round(4).to_string())
print("\n--- train acc by config x dataset ---")
print(res_b.pivot_table(index='config', columns='dataset',
                        values='final_train_acc', aggfunc='mean').round(4).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

a0_1conv16 | d90 | seed 11 | params 2,007,498 | train 900
    ep   1/15  loss 2.3990  acc 0.1433  val_loss 2.3111  val_acc 0.1000
    ep   5/15  loss 0.9074  acc 0.8011  val_loss 1.4509  val_acc 0.5150
    ep  10/15  loss 0.2328  acc 0.9767  val_loss 1.1392  val_acc 0.5850
    ep  15/15  loss 0.0981  acc 0.9989  val_loss 1.1167  val_acc 0.5900
  -> final_val 0.5900  best_val 0.6450  (33.9s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

a0_1conv16 | d90 | seed 22 | params 2,007,498 | train 900
    ep   1/15  loss 2.4756  acc 0.1300  val_loss 2.4603  val_acc 0.1100
    ep   5/15  loss 1.0245  acc 0.7611  val_loss 1.4349  val_acc 0.5650
    ep  10/15  loss 0.3301  acc 0.9456  val_loss 1.1611  val_acc 0.6200
    ep  15/15  loss 0.1053  acc 0.9989  val_loss 1.0723  val_acc 0.6600
  -> final_val 0.6600  best_val 0.6600  (33.1s)
Found 900 images belonging to 10 classes.


In [3]:
CONFIGS  = ["a0_1conv16","a1_1conv32","a2_2conv","a3_2conv_d128",
            "a4_baseline","a5_3conv","a6_3conv_narrow"]
DATASETS = ["d90","d138","d234"]
SEEDS    = [11, 22, 33]
EPOCHS   = 50

for name in CONFIGS:
    for dkey in DATASETS:
        for seed in SEEDS:
            r = subprocess.run(
                [sys.executable, "cnn_systematic_buildup_worker.py",
                 name, dkey, str(seed), str(EPOCHS)],
                capture_output=True, text=True, cwd=CWD)
            print(r.stdout.strip() or r.stderr.strip()[-600:])

rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/result_*.json"))]
res_b = pd.DataFrame(rows)
res_b.to_csv(f"{OUT}/buildup_log.csv", index=False)

for ep in sorted(res_b.epochs.unique()):
    sub = res_b[res_b.epochs == ep]
    print(f"\n===== {ep} EPOCHS =====")
    for metric in ['best_val_acc','final_val_acc','final_train_acc']:
        print(f"\n--- {metric} ---")
        print(sub.pivot_table(index='config', columns='dataset',
                              values=metric, aggfunc='mean').round(4).to_string())
    print("\n--- mean best_epoch (where val_loss was lowest) ---")
    print(sub.pivot_table(index='config', columns='dataset',
                          values='best_epoch', aggfunc='mean').round(1).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

a0_1conv16 | d90 | seed 11 | params 2,007,498 | train 900
    ep   1/50  loss 2.3990  acc 0.1433  val_loss 2.3111  val_acc 0.1000
    ep   5/50  loss 0.9074  acc 0.8011  val_loss 1.4509  val_acc 0.5150
    ep  10/50  loss 0.2328  acc 0.9767  val_loss 1.1393  val_acc 0.5850
    ep  15/50  loss 0.0981  acc 0.9989  val_loss 1.1168  val_acc 0.5900
    ep  20/50  loss 0.0465  acc 0.9989  val_loss 1.0653  val_acc 0.6350
    ep  25/50  loss 0.0252  acc 1.0000  val_loss 1.0746  val_acc 0.6200
    ep  30/50  loss 0.0159  acc 1.0000  val_loss 1.0654  val_acc 0.6200
    ep  35/50  loss 0.0107  acc 1.0000  val_loss 1.1176  val_acc 0.6350
    ep  40/50  loss 0.0072  acc 1.0000  val_loss 1.1820  val_acc 0.6450
    ep  45/50  loss 0.0219  acc 0.9967  val_loss 1.0918  val_acc 0.6750
    ep  50/50  loss 0.0041  acc 1.0000  val_loss 1.0957  val_acc 0.6450
  -> final_val 0.6450  best_val 0.6750  (100.9s)
Found 900 images

In [4]:
CONFIGS  = ["b0_2conv_d64","b1_2conv_d32","b2_2conv_d128_64","b3_2conv_d64_32",
            "b4_3conv_d64","b5_3conv_d128_64","b6_3conv_d64_32",
            "b7_4conv_d128","b8_4conv_d64"]
DATASETS = ["d90","d138","d234"]
SEEDS    = [11, 22, 33]
EPOCHS   = 30

for name in CONFIGS:
    for dkey in DATASETS:
        for seed in SEEDS:
            r = subprocess.run(
                [sys.executable, "cnn_systematic_buildup_worker.py",
                 name, dkey, str(seed), str(EPOCHS)],
                capture_output=True, text=True, cwd=CWD)
            print(r.stdout.strip() or r.stderr.strip()[-600:])

rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/result_*.json"))]
res_b = pd.DataFrame(rows)
res_b.to_csv(f"{OUT}/buildup_log.csv", index=False)

for ep in sorted(res_b.epochs.unique()):
    sub = res_b[res_b.epochs == ep]
    print(f"\n===== {ep} EPOCHS =====")
    for metric in ['best_val_acc','final_val_acc','final_train_acc']:
        print(f"\n--- {metric} ---")
        print(sub.pivot_table(index='config', columns='dataset',
                              values=metric, aggfunc='mean').round(4).to_string())
    print("\n--- mean best_epoch (where val_loss was lowest) ---")
    print(sub.pivot_table(index='config', columns='dataset',
                          values='best_epoch', aggfunc='mean').round(1).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

b0_2conv_d64 | d90 | seed 11 | params 12,865,162 | train 900
    ep   1/30  loss 2.5996  acc 0.1133  val_loss 2.4281  val_acc 0.1000
    ep   5/30  loss 0.5094  acc 0.9011  val_loss 1.2287  val_acc 0.5550
    ep  10/30  loss 0.0697  acc 1.0000  val_loss 1.1054  val_acc 0.6350
    ep  15/30  loss 0.0246  acc 1.0000  val_loss 1.0881  val_acc 0.6900
    ep  20/30  loss 0.0137  acc 1.0000  val_loss 1.0650  val_acc 0.6450
    ep  25/30  loss 0.0089  acc 1.0000  val_loss 1.0799  val_acc 0.6950
    ep  30/30  loss 0.0064  acc 1.0000  val_loss 1.0989  val_acc 0.6600
  -> final_val 0.6600  best_val 0.6950  (67.0s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

b0_2conv_d64 | d90 | seed 22 | params 12,865,162 | train 900
    ep   1/30  loss 2.6505  acc 0.0978  val_loss 2.3687  val_acc 0.1350
    ep   5/30  loss 0.5852  acc 0.8789  val_loss 1.1987  val_acc 0.6450
    ep  10/3

In [4]:
CONFIGS = ["c0_2conv_d128_64_32","c1_2conv_d64_32_16",
           "c2_2conv_wide_d128_64","c3_2conv_wide_d64_32",
           "c4_2conv_k5_d128_64","c5_2conv_k7_d128_64","c6_2conv_k5_d64_32",
           "c7_2conv_wide_k5","c8_2conv_d96_48"]
DATASETS = ["d90","d138","d234"]
SEEDS    = [11, 22, 33]
EPOCHS   = 30

for name in CONFIGS:
    for dkey in DATASETS:
        for seed in SEEDS:
            r = subprocess.run(
                [sys.executable, "cnn_systematic_buildup_worker.py",
                 name, dkey, str(seed), str(EPOCHS)],
                capture_output=True, text=True, cwd=CWD)
            print(r.stdout.strip() or r.stderr.strip()[-600:])

rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/result_*.json"))]
res_b = pd.DataFrame(rows)
res_b.to_csv(f"{OUT}/buildup_log.csv", index=False)

for ep in sorted(res_b.epochs.unique()):
    sub = res_b[res_b.epochs == ep]
    print(f"\n===== {ep} EPOCHS =====")
    for metric in ['best_val_acc','final_val_acc','final_train_acc']:
        print(f"\n--- {metric} ---")
        print(sub.pivot_table(index='config', columns='dataset',
                              values=metric, aggfunc='mean').round(4).to_string())
    print("\n--- mean best_epoch (where val_loss was lowest) ---")
    print(sub.pivot_table(index='config', columns='dataset',
                          values='best_epoch', aggfunc='mean').round(1).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

c0_2conv_d128_64_32 | d90 | seed 11 | params 25,720,298 | train 900
    ep   1/30  loss 2.4948  acc 0.1011  val_loss 2.3890  val_acc 0.1000
    ep   5/30  loss 1.1987  acc 0.7400  val_loss 1.4924  val_acc 0.5050
    ep  10/30  loss 0.2179  acc 1.0000  val_loss 1.0845  val_acc 0.6300
    ep  15/30  loss 0.1012  acc 1.0000  val_loss 1.0399  val_acc 0.6950
    ep  20/30  loss 0.0663  acc 1.0000  val_loss 1.0294  val_acc 0.6400
    ep  25/30  loss 0.0483  acc 1.0000  val_loss 1.0357  val_acc 0.6650
    ep  30/30  loss 0.0371  acc 1.0000  val_loss 1.0425  val_acc 0.6600
  -> final_val 0.6600  best_val 0.6950  (69.9s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

c0_2conv_d128_64_32 | d90 | seed 22 | params 25,720,298 | train 900
    ep   1/30  loss 2.5245  acc 0.0889  val_loss 2.4250  val_acc 0.1000
    ep   5/30  loss 1.5669  acc 0.5400  val_loss 1.6038  val_acc 0.490

In [3]:
CONFIGS  = [f"L2_{b}_{t}" for b in ["b2","b3","a4","c8","c4","b5","a5","b6","b8","a6"]
                          for t in ["1e5","1e4","1e3","1e2"]]
DATASETS = ["d90","d138","d234"]
SEEDS    = [11, 22, 33]
EPOCHS   = 25

for name in CONFIGS:
    for dkey in DATASETS:
        for seed in SEEDS:
            r = subprocess.run(
                [sys.executable, "cnn_systematic_buildup_worker.py",
                 name, dkey, str(seed), str(EPOCHS)],
                capture_output=True, text=True, cwd=CWD)
            print(r.stdout.strip() or r.stderr.strip()[-600:])

rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/result_*.json"))]
res_b = pd.DataFrame(rows)
res_b.to_csv(f"{OUT}/buildup_log.csv", index=False)

for ep in sorted(res_b.epochs.unique()):
    sub = res_b[res_b.epochs == ep]
    print(f"\n===== {ep} EPOCHS =====")
    for metric in ['best_val_acc','final_val_acc','final_train_acc']:
        print(f"\n--- {metric} ---")
        print(sub.pivot_table(index='config', columns='dataset',
                              values=metric, aggfunc='mean').round(4).to_string())
    print("\n--- mean best_epoch (where val_loss was lowest) ---")
    print(sub.pivot_table(index='config', columns='dataset',
                          values='best_epoch', aggfunc='mean').round(1).to_string())

Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

L2_b2_1e5 | d90 | seed 11 | params 25,718,538 | train 900
    ep   1/25  loss 2.6330  acc 0.0989  val_loss 2.4384  val_acc 0.1000
    ep   5/25  loss 0.6198  acc 0.8689  val_loss 1.1762  val_acc 0.6200
    ep  10/25  loss 0.0637  acc 1.0000  val_loss 0.9887  val_acc 0.7250
    ep  15/25  loss 0.0255  acc 1.0000  val_loss 0.9856  val_acc 0.7200
    ep  20/25  loss 0.0146  acc 1.0000  val_loss 0.9457  val_acc 0.7200
    ep  25/25  loss 0.0100  acc 1.0000  val_loss 0.9467  val_acc 0.7100
  -> final_val 0.7100  best_val 0.7500  (58.4s)
Found 900 images belonging to 10 classes.
Found 200 images belonging to 10 classes.

L2_b2_1e5 | d90 | seed 22 | params 25,718,538 | train 900
    ep   1/25  loss 2.6196  acc 0.0867  val_loss 2.3749  val_acc 0.1000
    ep   5/25  loss 1.1743  acc 0.6700  val_loss 1.4299  val_acc 0.5250
    ep  10/25  loss 0.0789  acc 1.0000  val_loss 1.0029  val_acc 0.6800
    ep  15/25  los

In [3]:
OUT_BT = "/home/admins/rebuild_workspace/models_buildup_test"
os.makedirs(OUT_BT, exist_ok=True)

for seed in [11, 22, 33]:
    r = subprocess.run([sys.executable, "cnn_systematic_buildup_test_worker.py", str(seed), "30"],
                       capture_output=True, text=True, cwd=CWD)
    print(r.stdout.strip() or r.stderr.strip()[-500:])

rows = [json.load(open(f)) for f in sorted(glob.glob(f"{OUT_BT}/result_*.json"))]
res_bt = pd.DataFrame(rows)
res_bt.to_csv(f"{OUT_BT}/buildup_test_log.csv", index=False)

print("\n--- b2 on d138 vs baseline ---")
print(f"                    baseline    b2/d138 (mean of 3)")
print(f"full test (200)      0.7150      {res_bt.test_full_acc.mean():.4f}  "
      f"({res_bt.test_full_acc.min():.4f}-{res_bt.test_full_acc.max():.4f})")
print(f"real only (90)       0.7556      {res_bt.test_real_acc.mean():.4f}  "
      f"({res_bt.test_real_acc.min():.4f}-{res_bt.test_real_acc.max():.4f})")

Found 1380 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
Found 90 images belonging to 10 classes.
seed 11  best_val 0.7600  final_val 0.7400  test_full 0.7200  test_real 0.7667  (93.3s)
Found 1380 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
Found 90 images belonging to 10 classes.
seed 22  best_val 0.8000  final_val 0.7550  test_full 0.7050  test_real 0.7000  (91.8s)
Found 1380 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
Found 200 images belonging to 10 classes.
Found 90 images belonging to 10 classes.
seed 33  best_val 0.7650  final_val 0.7400  test_full 0.7050  test_real 0.7222  (92.3s)

--- b2 on d138 vs baseline ---
                    baseline    b2/d138 (mean of 3)
full test (200)      0.7150      0.7100  (0.7050-0.7200)
real only (90)       0.7556      0.7296  (0.7000-0.7667)
